In [1]:
from neural_audio.examples.sounds.load import load_sound_file_paths
import soundfile as soundf
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Audio

# Get list of sound files 
sound_files = load_sound_file_paths()
sound_file_names = [Path(file).name for file in sound_files]

# Create dropwdown widget to select and play sound file
dropdown = widgets.Dropdown(options=sound_file_names, description='Sound file:')
audio_player = widgets.Output()
sound_file_path = sound_files[1]

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        global sound_file_path
        for file in sound_files:
            if Path(file).name == change['new']: sound_file_path = file
        audio_player.clear_output()
        with audio_player:
            display(Audio(str(sound_file_path), autoplay=False))

dropdown.observe(on_change)
dropdown.value = sound_file_names[3]  # Set default value to the first sound file
display(dropdown, audio_player)


Dropdown(description='Sound file:', index=3, options=('cardboard tapping.wav', 'cardboard whirling.wav', 'cric…

Output()

In [2]:
from neural_audio.wav2aud import wav2aud
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

waveform, sampling_rate = soundf.read(sound_file_path)
time_points_audiogram, frequencies_audiogram, audiogram  = wav2aud(waveform)

# Convert the magnitude to dB safely
to_decibel = lambda x: 20 * np.log10(np.maximum(x, 1e-9))  # Avoid log of zero by adding a small constant

plot = False

def plot_matrix(matrix, time_points, frequencies, ylim, title):
    plt.title(title)

    # Use pcolormesh so the linear FFT bins can be mapped to a log scale natively
    plt.pcolormesh(time_points, frequencies, to_decibel(matrix))
    plt.colorbar(label='Magnitude (dB)')

    # Axes labels and scaling
    plt.xlabel("Time (s)")
    plt.yscale('log', base=2)
    plt.ylim(*ylim)  # Limit y-axis to human hearing range
    plt.ylabel("Frequency (Hz)")
    plt.gca().yaxis.set_major_formatter(ticker.ScalarFormatter())

if plot:
    plt.figure(figsize=(12, 6))
    # Bottom plot: audiogram
    plot_matrix(matrix=audiogram, time_points=time_points_audiogram, frequencies=frequencies_audiogram, ylim=[frequencies_audiogram[0], frequencies_audiogram[-1]], title="Audiogram")

    plt.tight_layout()
    plt.show()

In [3]:
from neural_audio.aud2cor import aud2cor
sf = 16000
para1 = [4, 0, -2, np.log2(sf/16000), 0, 0, 1]
rv = 2 ** np.linspace(np.log2(0.5), np.log2(128), 32)
sv = 2 ** np.linspace(np.log2(1/5), np.log2(10), 32)

cr = aud2cor(audiogram.T, para1, rv, sv, 'tmp', DISP=0)

In [4]:
print(cr.shape)
print(cr[0, 0, :5, :5])       # first scale, first rate, top-left corner


(32, 64, 750, 128)
[[4.17705093e-07-1.27957729e-06j 4.82472848e-07-1.29503581e-06j
  5.49872839e-07-1.30719880e-06j 6.19750579e-07-1.31579880e-06j
  6.91926354e-07-1.32057310e-06j]
 [4.19719378e-07-1.28818451e-06j 4.84906213e-07-1.30378567e-06j
  5.52744999e-07-1.31607110e-06j 6.23080403e-07-1.32477146e-06j
  6.95731697e-07-1.32962215e-06j]
 [4.21629262e-07-1.29664597e-06j 4.87224957e-07-1.31239017e-06j
  5.55492005e-07-1.32479907e-06j 6.26274260e-07-1.33360147e-06j
  6.99390015e-07-1.33853093e-06j]
 [4.23553000e-07-1.30506455e-06j 4.89555379e-07-1.32094936e-06j
  5.58248276e-07-1.33347937e-06j 6.29474733e-07-1.34238155e-06j
  7.03052065e-07-1.34738763e-06j]
 [4.25531492e-07-1.31348296e-06j 4.91941627e-07-1.32950551e-06j
  5.61061268e-07-1.34215361e-06j 6.32732630e-07-1.35115240e-06j
  7.06772031e-07-1.35623182e-06j]]


In [5]:
print(np.abs(cr).max())        # global max magnitude
print(np.abs(cr[0, 0]).max())  # max for one slice

0.0008676406019768458
2.9531526873490222e-05


In [9]:
from pathlib import Path
import h5py

matlab_outputs = Path("C:/Users/carol/OneDrive/Documents/UMaastricht/AudioToolbox/matlab/matlab_outputs/aud2cor")

current = matlab_outputs / "cr_glass rubbing.mat"
with h5py.File(current, "r") as f:
    cr_matlab = f["cr"][:]


In [ ]:
cr_matlab_t = cr_matlab.T
cr_matlab_t.shape

In [16]:
cr_matlab_complex = (
    cr_matlab_t["real"] +
    1j * cr_matlab_t["imag"]
)

In [17]:
print(np.max(np.abs(cr_matlab_complex - cr)))

7.4064837310717815e-06


In [19]:
print(np.linalg.norm(cr_matlab_complex - cr))
print(np.linalg.norm(cr_matlab_complex))
rel = np.linalg.norm(cr_matlab_complex - cr) / np.linalg.norm(cr_matlab_complex)
print(rel)

0.0007476324046465812
0.3084800894051507
0.0024236001943861536
